In [1]:
import os
import datetime
from datetime import datetime
from datetime import timedelta
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 22})
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.colors as colors
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cbook as cbook
import netCDF4
import xarray as xr
import pandas as pd
import glob
from typing import List
%matplotlib inline
from mpl_toolkits.axes_grid1 import make_axes_locatable
import warnings
warnings.simplefilter('ignore')

In [5]:
# # GET FILE
dfile_1 = '/data/ref/sa1.ld.z01.00/2023/07/03/sa1.ld.z01.00.20230703.190045.txt'# Rain present
dfile_2 = '/data/ref/sa1.ld.z01.00/2023/07/03/sa1.ld.z01.00.20230703.205945.txt'# Clear
dfile_3 = '/home/goldberger/AWAKEN/sa1.ld.z01.00.20230701.000030.txt'


In [3]:
# os.path.isfile('/data/share/goldberger/sa1.ld/20221117.nc')

In [3]:
# DEFINE BIN SIZES + FORMAT
D_ve_avg = [0.062,0.187,0.312,0.437,0.562,0.687,0.812,0.937,
            1.062,1.187,1.375,1.625,1.875,2.125,2.375,2.75,
            3.25,3.75,4.25,4.75,5.5,6.5,7.5,8.5,9.5,11.,13.,
            15.,17.,19.,21.5,24.5]
D_ve_spread = [0.125,0.125,0.125,0.125,0.125,0.125,0.125,
               0.125,0.125,0.125,0.25,0.25,0.25,0.25,
               0.25,0.5,0.5,0.5,0.5,0.5,1.,1.,1.,1.,1.,2.,2.,
               2.,2.,2.,3.,3.]
Speed_ms_avg = [0.05,0.15,0.25,0.35,0.45,0.55,0.65,0.75,0.85,
                0.95,1.1,1.3,1.5,1.7,1.9,2.2,2.6,3.0,3.4,3.8,
                4.4,5.2,6.0,6.8,7.6,8.8,10.4,12.,13.6,15.2,17.6,20.8]
Speed_ms_spread = [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,
                  0.2,0.2,0.2,0.2,0.2,0.4,0.4,0.4,0.4,0.4,
                  0.8,0.8,0.8,0.8,0.8,1.6,1.6,1.6,1.6,1.6,
                  3.2,3.2]
D_ve_avg = np.reshape(np.array(D_ve_avg), (32,))
Speed_ms_avg = np.reshape(np.array(Speed_ms_avg), (32,))
D_ve_spread = np.reshape(np.array(D_ve_spread), (32,))
Speed_ms_spread = np.reshape(np.array(Speed_ms_spread), (32,))

In [ ]:
# Open a Single File
def load_disdrometer_txt_file(dfile):
    # READ FILE
    DAT = pd.read_csv(dfile, sep=",", header = None)

    # GET TIME
    time = pd.to_datetime(dfile[-19:-4],format='%Y%m%d.%H%M%S', errors='ignore').to_numpy()
    time = np.reshape(np.array(time), (1,))
    # PULL OUT VARIABLES + FORMAT
    radar_reflectivity_dBz = float(DAT[0][6][3:]); radar_reflectivity_dBz = np.reshape(np.array(radar_reflectivity_dBz), (1,))
    MOR_visibility_m = float(DAT[0][7][3:]); MOR_visibility_m = np.reshape(np.array(MOR_visibility_m), (1,))
    serial_num = float(DAT[0][12][3:]); serial_num = np.reshape(np.array(serial_num), (1,))
    row_90 = 10**(np.array([float(x) for x in DAT[0][81][3:].split(';') if x.strip()]))
    row_90 = row_90.astype(int)
    row_90 = np.reshape(np.array(row_90), (1,32))
    # row 90 #/m3 per diameter number density distribution, N(d) 
    row_91 = np.round(np.array([float(x) for x in DAT[0][82][3:].split(';') if x.strip()]),0).astype(int)
    row_91 = np.reshape(np.array(row_91), (1,32))
    # row 91, fall speed distribution (?) fall speed = f(diameter)
    # row 93 = 'raw data' d_ve, 93 is the 2D data (should be 32,32)
    row_93 = np.array([float(x) for x in DAT[0][83][3:].split(';') if x.strip()])
    row_93 = np.reshape(np.array(row_93), (32,32))
    row_93_sd = np.expand_dims(row_93, axis=0) #add singleton dimension
    rain_intensity_mmph = float(DAT[0][0][3:]); rain_intensity_mmph = np.reshape(np.array(rain_intensity_mmph), (1,))
    rain_accumulated_mm = float(DAT[0][1][3:]); rain_accumulated_mm = np.reshape(np.array(rain_accumulated_mm), (1,))
    num_particles = int(DAT[0][10][3:]); num_particles = np.reshape(np.array(num_particles), (1,))
    NWS_weather_code = DAT.iloc[5][0][3:].lstrip(" "); NWS_weather_code = np.reshape(np.array(NWS_weather_code), (1,))

    # CREATE XARRAY DATASET
    # Create coordinates
    coords = {'time': time, 'diameter_ve': D_ve_avg, 'fall_Speed_ms': Speed_ms_avg}
    # Define the dimension names
    dims = ('time', 'diameter_ve', 'fall_Speed_ms')
    # Create xarray dataset
    ds = xr.Dataset(
        {
            'radar_reflectivity_dBz': (['time'], radar_reflectivity_dBz),  
            'MOR_visibility_m': (['time'], MOR_visibility_m),
            'serial_num': (['time'], serial_num),
            'rain_intensity_mmph': (['time'], rain_intensity_mmph),
            'rain_accumulated_mm': (['time'], rain_accumulated_mm),
            'num_particles': (['time'], num_particles),
            'NWS_weather_code': (['time'], NWS_weather_code),
            'Dve_bin_concentration': (['time', 'diameter_ve'], row_90),
            'FS_bin_concentration': (['time', 'fall_Speed_ms'], row_91),
            'Dve_FS_bin_concentration': (['time','fall_Speed_ms','diameter_ve'], row_93_sd),
            'D_ve_spread': (['diameter_ve'], D_ve_spread),
            'Speed_ms_spread': (['fall_Speed_ms'], Speed_ms_spread),
        },
        coords=coords
    )
    return ds

# ds = load_disdrometer_txt_file(dfile)


In [22]:
# ds.to_netcdf('/home/goldberger/AWAKEN/disdrometer_tmp.nc')

In [6]:
# file_paths = [dfile_1,dfile_2]
# datasets = []

# for file_path in file_paths:
#     dataset = load_disdrometer_txt_file(file_path)
#     datasets.append(dataset)

# combined_dataset = xr.concat(datasets, dim='time')
# combined_dataset.to_netcdf('/data/share/goldberger/combined_dataset_test.nc')



In [7]:
# # Define the root directory
# root_dir = '/data/ref/sa1.ld.z01.00'
# year = '2023'
# month = '07'
# day = '03'

# # Use glob to get all the .txt files in subfolders
# txt_files = glob.glob(os.path.join(root_dir,year,month,day, '**/*.txt'), recursive=True)
# file_paths = sorted(txt_files)

# datasets = []

# for file_path in file_paths:
#     dataset = load_disdrometer_txt_file(file_path)
#     datasets.append(dataset)

# combined_dataset = xr.concat(datasets, dim='time')
# # combined_dataset.to_netcdf('/path/to/save/directory/combined_dataset.nc')

In [8]:
# year_path = os.path.join(root_dir,os.listdir(root_dir)[0])
# month_path = os.path.join(year_path, os.listdir(year_path)[0])
# day_path = os.path.join(month_path, os.listdir(month_path)[0])
# daily_files = [os.path.join(day_path, file) for file in os.listdir(day_path) if file.endswith('.txt')]

# file_paths = sorted(daily_files)

# datasets = []

# for file_path in file_paths:
#     dataset = load_disdrometer_txt_file(file_path)
#     datasets.append(dataset)

# combined_dataset = xr.concat(datasets, dim='time')

# combined_dataset.to_netcdf('/home/goldberger/AWAKEN/disdrometer_netcdf_files/sa1' + file_paths[0][-19:-11] + '.nc')

In [9]:
# combined_dataset

In [12]:
# os.listdir(root_dir)

['2022', '2023']

In [5]:
# Define the root directory
# root_dir = '/data/ref/sa1.ld.z01.00'
# save_dir = '/data/share/goldberger/sa1.ld/'
#
# root_dir = '/data/ref/sat.ld.z01.00'
# save_dir = '/data/share/goldberger/sat.ld/'

root_dir = '/data/ref/sa1.ld.z02.00'
save_dir = '/data/share/goldberger/sa1.ld.z02/'

# Iterate through year directories
for year_dir in sorted(os.listdir(root_dir)):
    year_path = os.path.join(root_dir, year_dir)
    if not os.path.isdir(year_path):
        continue
    # Iterate through month directories
    for month_dir in sorted(os.listdir(year_path)):
        month_path = os.path.join(year_path, month_dir)
        if not os.path.isdir(month_path):
            continue
        # Iterate through month directories
        for day_dir in sorted(os.listdir(month_path)):
            day_path = os.path.join(month_path, day_dir)
            if not os.path.isdir(day_path):
                continue
            # Get all the text files in the day directory
            daily_files = [os.path.join(day_path, file) for file in os.listdir(day_path) if file.endswith('.txt')]
            if daily_files:
                file_paths = sorted(daily_files)
                fnam = save_dir + file_paths[0][-19:-11] + '.nc'
                # # Commenting out if-else statement to regenerate files
                # if os.path.isfile(fnam):
                #     print('exists: ' + fnam)
                #     pass
                # else:
                datasets = []
                for file_path in file_paths:
                    dataset = load_disdrometer_txt_file(file_path)
                    datasets.append(dataset)
                combined_dataset = xr.concat(datasets, dim='time')
                combined_dataset.to_netcdf(fnam)
                print('created: ' + fnam)

created: /data/share/goldberger/sa1.ld.z02/20230317.nc
created: /data/share/goldberger/sa1.ld.z02/20230318.nc
created: /data/share/goldberger/sa1.ld.z02/20230319.nc
created: /data/share/goldberger/sa1.ld.z02/20230320.nc
created: /data/share/goldberger/sa1.ld.z02/20230321.nc
created: /data/share/goldberger/sa1.ld.z02/20230322.nc
created: /data/share/goldberger/sa1.ld.z02/20230412.nc
created: /data/share/goldberger/sa1.ld.z02/20230413.nc
created: /data/share/goldberger/sa1.ld.z02/20230414.nc
created: /data/share/goldberger/sa1.ld.z02/20230415.nc
created: /data/share/goldberger/sa1.ld.z02/20230416.nc
created: /data/share/goldberger/sa1.ld.z02/20230417.nc
created: /data/share/goldberger/sa1.ld.z02/20230418.nc
created: /data/share/goldberger/sa1.ld.z02/20230419.nc
created: /data/share/goldberger/sa1.ld.z02/20230420.nc
created: /data/share/goldberger/sa1.ld.z02/20230421.nc
created: /data/share/goldberger/sa1.ld.z02/20230422.nc
created: /data/share/goldberger/sa1.ld.z02/20230423.nc
created: /

In [13]:
# root_dir = '/data/ref/sa1.ld.z01.00'
# sorted(os.listdir(root_dir))
# # year_path = os.path.join(root_dir, '2023')
# # print(year_path)
# # print(os.listdir(year_path))
# # sorted(os.listdir(year_path))

['2022', '2023']

In [16]:
# # Define the root directory
# root_dir = '/data/ref/sa1.ld.z01.00'
# save_dir = '/data/share/goldberger/sa1.ld/'
# year_path = os.path.join(root_dir, '2023')
# # Iterate through year directories
# # for year_dir in os.listdir(root_dir):
# #     year_path = os.path.join(root_dir, year_dir)
# #     if not os.path.isdir(year_path):
# #         continue
# for year_dir in year_path:
#     # Iterate through month directories
#     for month_dir in os.listdir(year_path):
#         month_path = os.path.join(year_path, month_dir)
#         if not os.path.isdir(month_path):
#             continue
#         # Iterate through month directories
#         for day_dir in os.listdir(month_path):
#             day_path = os.path.join(month_path, day_dir)
#             if not os.path.isdir(day_path):
#                 continue
#             # Get all the text files in the day directory
#             daily_files = [os.path.join(day_path, file) for file in os.listdir(day_path) if file.endswith('.txt')]
#             if daily_files:
#                 file_paths = sorted(daily_files)
#                 fnam = save_dir + file_paths[0][-19:-11] + '.nc'
#                 if os.path.isfile(fnam):
#                     print('exists: ' + fnam)
#                     pass
#                 else:
#                     datasets = []
#                     for file_path in file_paths:
#                         dataset = load_disdrometer_txt_file(file_path)
#                         datasets.append(dataset)
#                     combined_dataset = xr.concat(datasets, dim='time')
#                     combined_dataset.to_netcdf(fnam)
#                     print('created: ' + fnam)

created: /data/share/goldberger/sa1.ld/20230826.nc
created: /data/share/goldberger/sa1.ld/20230808.nc
created: /data/share/goldberger/sa1.ld/20230817.nc
created: /data/share/goldberger/sa1.ld/20230822.nc
created: /data/share/goldberger/sa1.ld/20230804.nc
created: /data/share/goldberger/sa1.ld/20230813.nc
created: /data/share/goldberger/sa1.ld/20230820.nc
created: /data/share/goldberger/sa1.ld/20230802.nc
created: /data/share/goldberger/sa1.ld/20230811.nc
created: /data/share/goldberger/sa1.ld/20230828.nc
created: /data/share/goldberger/sa1.ld/20230819.nc
created: /data/share/goldberger/sa1.ld/20230824.nc
created: /data/share/goldberger/sa1.ld/20230806.nc
created: /data/share/goldberger/sa1.ld/20230815.nc
created: /data/share/goldberger/sa1.ld/20230801.nc
created: /data/share/goldberger/sa1.ld/20230810.nc
created: /data/share/goldberger/sa1.ld/20230827.nc
created: /data/share/goldberger/sa1.ld/20230809.nc
created: /data/share/goldberger/sa1.ld/20230818.nc
created: /data/share/goldberger